
# Detecção de estrelas em 30 Doradus / R136 com YOLO

Este notebook constrói um **pipeline científico inicial** para detectar fontes estelares em um campo denso da Nebulosa da Tarântula (30 Doradus), centrado no aglomerado **R136**.

### Objetivos

1. Obter um **cutout FITS real do Hubble** via MAST/HAPCut.
2. Consultar o catálogo fotométrico do **Hubble Tarantula Treasury Project (HTTP)** via VizieR.
3. Converter RA/Dec das estrelas catalogadas para coordenadas de pixel usando o WCS da imagem.
4. Gerar automaticamente imagens e labels no formato YOLO.
5. Treinar um detector YOLO11 para uma única classe: `star`.
6. Avaliar não apenas mAP, mas também **precision, recall/completeness e erro posicional**.
7. Medir a **completude em função da magnitude F555W**.

> **Importante:** este é um experimento piloto. O objetivo é validar o pipeline e entender os limites do YOLO em campos congestionados. Para um resultado publicável, o passo seguinte é ampliar o conjunto de campos e fazer separação espacial rigorosa entre treino, validação e teste.

### Dados científicos

- **R136 / 30 Doradus:** RA = 05h 38m 42.39s, Dec = −69° 06′ 02.81″ (NASA/STScI).
- **HTTP / MAST:** Hubble Tarantula Treasury Project, DOI `10.17909/T9RP4V`.
- **Catálogo VizieR:** `J/ApJS/222/11`, Sabbi et al. (2016), DOI VizieR `10.26093/cds/vizier.22220011`.
- O catálogo HTTP contém mais de 800 mil fontes e fotometria HST em F275W, F336W, F555W, F658N, F775W, F110W e F160W.

### Nota metodológica

O catálogo HTTP foi produzido com ajuste de PSF. Neste notebook usamos preferencialmente fontes com `f_F555mag == 1`, isto é, fontes bem ajustadas pelo modelo de PSF e classificadas como prováveis estrelas pelo catálogo.

O YOLO não foi originalmente criado para fotometria estelar. Aqui ele será tratado como um **detector de fontes pontuais baseado em Deep Learning**, e seu desempenho deverá ser comparado posteriormente com métodos como DAOStarFinder e PSF fitting.


In [ ]:

# ============================================================
# 1. Instalação das dependências
# Execute esta célula no Google Colab.
# ============================================================

!pip -q install astroquery astropy photutils ultralytics scipy pyyaml


In [ ]:

# ============================================================
# 2. Imports e configuração
# ============================================================

import os
import math
import shutil
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
import yaml

import astropy.units as u
from astropy.coordinates import SkyCoord
from astropy.io import fits
from astropy.wcs import WCS
from astropy.visualization import PercentileInterval, AsinhStretch

from astroquery.mast import Hapcut
from astroquery.vizier import Vizier

from scipy.spatial import cKDTree

SEED = 42
rng = np.random.default_rng(SEED)

ROOT = Path("/content/tarantula_yolo")
ROOT.mkdir(parents=True, exist_ok=True)

print("Diretório de trabalho:", ROOT)



## 3. Região de estudo: R136

R136 é o núcleo extremamente denso de NGC 2070, dentro de 30 Doradus. Ele funciona como nosso **stress test** para detecção de estrelas em regiões congestionadas.

Começaremos com um cutout de aproximadamente 120 arcsec de lado. Para um HST óptico com escala próxima de 0,04 arcsec/pixel, isso corresponde a alguns milhares de pixels por lado — suficiente para formar diversos tiles de treinamento sem baixar mosaicos completos de vários gigabytes.


In [ ]:

# ============================================================
# 3. Coordenadas e parâmetros principais
# ============================================================

R136 = SkyCoord(
    "05h38m42.39s",
    "-69d06m02.81s",
    frame="icrs"
)

CUTOUT_SIZE = 120 * u.arcsec

# Parâmetros do dataset
TILE_SIZE = 512
VAL_FRACTION = 0.20

# Catálogo: limite inicial para evitar densidade extrema de labels.
# Depois podemos testar 24, 25, 26, 27... e construir curvas de completude.
MAG_LIMIT = 25.5

# Caixa fixa centrada na fonte. Em HST óptico, a PSF ocupa poucos pixels.
BOX_SIZE_PX = 10.0

# Avaliação astrométrica
MATCH_RADIUS_PX = 5.0

print("R136:", R136.to_string("hmsdms"))
print("Cutout:", CUTOUT_SIZE)



## 4. Download de um cutout FITS real do Hubble

Usamos o serviço **HAPCut do MAST**, que fornece recortes de Hubble Advanced Products sem exigir o download dos mosaicos completos.

A consulta pode retornar mais de um produto/filtro. O código abaixo procura primeiro por **F555W**, por ser uma das bandas presentes no catálogo HTTP. Se F555W não estiver disponível no conjunto retornado, escolhe um produto óptico como fallback e informa qual foi usado.


In [ ]:

# ============================================================
# 4. Obter cutouts HST reais via MAST HAPCut
# ============================================================

cutouts = Hapcut.get_cutouts(
    coordinates=R136,
    size=CUTOUT_SIZE
)

if not cutouts:
    raise RuntimeError(
        "O HAPCut não retornou imagens para esta posição. "
        "Tente novamente mais tarde ou reduza CUTOUT_SIZE."
    )

print(f"Produtos HST retornados: {len(cutouts)}")


def get_science_hdu(hdul):
    # HAPCut normalmente fornece uma extensão SCI.
    if "SCI" in hdul:
        return hdul["SCI"]

    # Fallback: primeira extensão que contenha uma imagem 2D.
    for hdu in hdul:
        if getattr(hdu, "data", None) is not None:
            arr = np.asarray(hdu.data)
            if arr.ndim == 2:
                return hdu

    raise ValueError("Nenhuma imagem 2D encontrada no HDUList.")


def describe_hdul(hdul, idx):
    sci = get_science_hdu(hdul)
    headers = [hdul[0].header, sci.header]

    text_parts = []
    for header in headers:
        for key in [
            "FILENAME", "FILTER", "FILTER1", "FILTER2",
            "INSTRUME", "DETECTOR", "PROPOSID", "TARGNAME"
        ]:
            if key in header:
                text_parts.append(f"{key}={header[key]}")

    text = " | ".join(text_parts)
    shape = np.asarray(sci.data).shape
    print(f"[{idx}] shape={shape} | {text}")
    return text.upper()


descriptions = []
for i, hdul in enumerate(cutouts):
    descriptions.append(describe_hdul(hdul, i))


In [ ]:

# ============================================================
# 5. Selecionar o produto mais apropriado
# ============================================================

# Preferência por filtros ópticos próximos do F555W do catálogo HTTP.
preferences = ["F555W", "F606W", "F775W", "F814W", "F475W", "F336W"]

selected_idx = None
selected_filter = None

for filt in preferences:
    matches = [i for i, desc in enumerate(descriptions) if filt in desc]
    if matches:
        selected_idx = matches[0]
        selected_filter = filt
        break

if selected_idx is None:
    # Fallback para o primeiro produto HST retornado.
    selected_idx = 0
    selected_filter = "desconhecido"

hdul = cutouts[selected_idx]
sci_hdu = get_science_hdu(hdul)

data = np.asarray(sci_hdu.data, dtype=np.float32)
wcs = WCS(sci_hdu.header).celestial

print("Produto escolhido:", selected_idx)
print("Filtro identificado:", selected_filter)
print("Shape:", data.shape)
print("Pixels finitos:", np.isfinite(data).sum())

# Salvar uma cópia FITS do recorte para rastreabilidade.
fits_path = ROOT / "r136_hubble_cutout.fits"
hdul.writeto(fits_path, overwrite=True)
print("FITS salvo em:", fits_path)


In [ ]:

# ============================================================
# 6. Visualizar o FITS com stretch astronômico
# ============================================================

finite = data[np.isfinite(data)]

interval = PercentileInterval(99.7)
vmin, vmax = interval.get_limits(finite)

scaled = np.clip((data - vmin) / (vmax - vmin + 1e-12), 0, 1)
scaled = AsinhStretch(a=0.05)(scaled)
scaled = np.nan_to_num(scaled, nan=0.0, posinf=1.0, neginf=0.0)

plt.figure(figsize=(10, 10))
plt.imshow(scaled, origin="lower", cmap="gray")
plt.title(f"R136 / 30 Doradus — HST cutout ({selected_filter})")
plt.xlabel("x [pixel]")
plt.ylabel("y [pixel]")
plt.show()



## 7. Catálogo fotométrico HTTP via VizieR

Em vez de baixar o catálogo completo de mais de 800 mil fontes, consultamos somente uma região ao redor de R136.

Campos utilizados:

- `RAdeg`, `DEdeg`: coordenadas J2000 em graus.
- `F555mag`: magnitude HST/ACS F555W.
- `f_F555mag`: flag de qualidade fotométrica.
- `HTTP`: identificador da fonte.

A flag `f_F555mag = 1` indica uma fonte bem ajustada pela PSF e provavelmente estelar.


In [ ]:

# ============================================================
# 7. Consultar o catálogo HTTP ao redor de R136
# ============================================================

Vizier.ROW_LIMIT = -1

viz = Vizier(
    columns=[
        "HTTP",
        "RAdeg",
        "DEdeg",
        "F555mag",
        "e_F555mag",
        "q_F555mag",
        "f_F555mag",
    ]
)

# O raio é um pouco maior que metade da diagonal do cutout.
catalog_radius = 100 * u.arcsec

tables = viz.query_region(
    R136,
    radius=catalog_radius,
    catalog="J/ApJS/222/11/photcat"
)

if len(tables) == 0:
    raise RuntimeError("Nenhuma fonte HTTP encontrada no VizieR.")

catalog = tables[0]

print("Fontes retornadas pelo VizieR:", len(catalog))
print("Colunas:", catalog.colnames)
catalog[:5]


In [ ]:

# ============================================================
# 8. Seleção das estrelas usadas como ground truth inicial
# ============================================================

cat = catalog.to_pandas()

# Converter campos numéricos explicitamente.
for col in ["RAdeg", "DEdeg", "F555mag", "e_F555mag", "q_F555mag", "f_F555mag"]:
    if col in cat.columns:
        cat[col] = pd.to_numeric(cat[col], errors="coerce")

selection = (
    np.isfinite(cat["RAdeg"]) &
    np.isfinite(cat["DEdeg"]) &
    np.isfinite(cat["F555mag"]) &
    (cat["F555mag"] < 90) &
    (cat["F555mag"] <= MAG_LIMIT) &
    (cat["f_F555mag"] == 1)
)

stars = cat.loc[selection].copy().reset_index(drop=True)

print("Fontes com flag F555W=1 e magnitude <=", MAG_LIMIT, ":", len(stars))
stars.head()


In [ ]:

# ============================================================
# 9. Converter RA/Dec -> pixels da imagem HST
# ============================================================

sky = SkyCoord(
    stars["RAdeg"].to_numpy() * u.deg,
    stars["DEdeg"].to_numpy() * u.deg,
    frame="icrs",
)

x, y = wcs.world_to_pixel(sky)

stars["x"] = x
stars["y"] = y

height, width = data.shape

inside = (
    np.isfinite(stars["x"]) &
    np.isfinite(stars["y"]) &
    (stars["x"] >= 0) &
    (stars["x"] < width) &
    (stars["y"] >= 0) &
    (stars["y"] < height)
)

stars_img = stars.loc[inside].copy().reset_index(drop=True)

print("Estrelas catalogadas dentro do cutout:", len(stars_img))
print("Densidade média aproximada:",
      len(stars_img) / (width * height) * 1e6,
      "estrelas por milhão de pixels")



## 10. Sanity check astrométrico

Antes de criar qualquer label, precisamos verificar visualmente se as posições do catálogo coincidem com as fontes da imagem.

Se os círculos aparecerem sistematicamente deslocados, **não prossiga diretamente para o treinamento**. Isso indicaria um offset astrométrico entre o produto HAP e o catálogo HTTP. Nesse caso, o próximo passo é calcular uma correção global de WCS/offset usando fontes brilhantes.


In [ ]:

# ============================================================
# 10. Sobrepor catálogo e imagem
# ============================================================

plt.figure(figsize=(12, 12))
plt.imshow(scaled, origin="lower", cmap="gray")

# Para não esconder completamente a imagem em campos muito densos,
# mostramos no máximo 5000 marcadores nesta visualização.
plot_stars = stars_img
if len(plot_stars) > 5000:
    plot_stars = plot_stars.sample(5000, random_state=SEED)

plt.scatter(
    plot_stars["x"],
    plot_stars["y"],
    s=18,
    facecolors="none",
    edgecolors="tab:red",
    linewidths=0.5,
    alpha=0.65,
)

plt.title(
    f"HTTP sobre HST — R136 | F555W <= {MAG_LIMIT} | "
    f"{len(stars_img)} fontes dentro do cutout"
)
plt.xlabel("x [pixel]")
plt.ylabel("y [pixel]")
plt.show()



## 11. Criar tiles e labels YOLO

Usaremos tiles de `512 × 512`, evitando reduzir drasticamente a imagem.

Para cada estrela catalogada criamos uma pequena caixa centrada na posição da fonte. Neste primeiro teste todas as caixas têm o mesmo tamanho. Isso é uma simplificação: em um estudo posterior podemos definir o tamanho da caixa a partir da PSF, FWHM ou magnitude.

O formato de cada label YOLO é:

```text
class_id x_center y_center width height
```

com coordenadas normalizadas entre 0 e 1.


In [ ]:

# ============================================================
# 11. Preparar diretórios
# ============================================================

for split in ["train", "val"]:
    (ROOT / "images" / split).mkdir(parents=True, exist_ok=True)
    (ROOT / "labels" / split).mkdir(parents=True, exist_ok=True)
    (ROOT / "metadata" / split).mkdir(parents=True, exist_ok=True)

# Limpar arquivos de execuções anteriores.
for subdir in ["images/train", "images/val", "labels/train", "labels/val",
               "metadata/train", "metadata/val"]:
    p = ROOT / subdir
    for f in p.glob("*"):
        if f.is_file():
            f.unlink()


In [ ]:

# ============================================================
# 12. Gerar a imagem de 8 bits preservando o stretch global
# ============================================================

image8 = np.clip(scaled * 255, 0, 255).astype(np.uint8)

# YOLO aceita PNG grayscale, mas replicaremos em RGB para compatibilidade.
image_rgb = np.repeat(image8[..., None], 3, axis=2)

print("Imagem pronta:", image_rgb.shape, image_rgb.dtype)


In [ ]:

# ============================================================
# 13. Gerar a lista de tiles completos
# ============================================================

tiles = []

tile_id = 0
for y0 in range(0, height - TILE_SIZE + 1, TILE_SIZE):
    for x0 in range(0, width - TILE_SIZE + 1, TILE_SIZE):
        x1 = x0 + TILE_SIZE
        y1 = y0 + TILE_SIZE

        nstars = int(
            (
                (stars_img["x"] >= x0) &
                (stars_img["x"] < x1) &
                (stars_img["y"] >= y0) &
                (stars_img["y"] < y1)
            ).sum()
        )

        tiles.append({
            "tile_id": tile_id,
            "x0": x0,
            "y0": y0,
            "x1": x1,
            "y1": y1,
            "nstars": nstars,
        })
        tile_id += 1

tiles_df = pd.DataFrame(tiles)

if len(tiles_df) < 5:
    raise RuntimeError(
        f"Apenas {len(tiles_df)} tiles completos foram gerados. "
        "Aumente CUTOUT_SIZE ou reduza TILE_SIZE."
    )

print("Tiles completos:", len(tiles_df))
print(tiles_df["nstars"].describe())


In [ ]:

# ============================================================
# 14. Separação treino/validação por tiles não sobrepostos
# ============================================================

indices = np.arange(len(tiles_df))
rng.shuffle(indices)

n_val = max(1, int(round(len(indices) * VAL_FRACTION)))
val_idx = set(indices[:n_val])

tiles_df["split"] = [
    "val" if i in val_idx else "train"
    for i in range(len(tiles_df))
]

print(tiles_df.groupby("split")["nstars"].agg(["count", "sum", "mean", "min", "max"]))


In [ ]:

# ============================================================
# 15. Funções de criação dos labels
# ============================================================

def clip_box(cx, cy, box_size, image_size):
    half = box_size / 2.0

    x1 = max(0.0, cx - half)
    y1 = max(0.0, cy - half)
    x2 = min(float(image_size), cx + half)
    y2 = min(float(image_size), cy + half)

    w = max(0.0, x2 - x1)
    h = max(0.0, y2 - y1)

    cx2 = (x1 + x2) / 2.0
    cy2 = (y1 + y2) / 2.0

    return cx2, cy2, w, h


def yolo_line(cx, cy, w, h, image_size):
    return (
        f"0 "
        f"{cx / image_size:.6f} "
        f"{cy / image_size:.6f} "
        f"{w / image_size:.6f} "
        f"{h / image_size:.6f}"
    )


In [ ]:

# ============================================================
# 16. Escrever imagens, labels e metadados
# ============================================================

records = []

for _, tile in tiles_df.iterrows():
    split = tile["split"]
    x0, y0 = int(tile["x0"]), int(tile["y0"])
    x1, y1 = int(tile["x1"]), int(tile["y1"])
    tid = int(tile["tile_id"])

    crop = image_rgb[y0:y1, x0:x1]

    tile_stars = stars_img[
        (stars_img["x"] >= x0) &
        (stars_img["x"] < x1) &
        (stars_img["y"] >= y0) &
        (stars_img["y"] < y1)
    ].copy()

    tile_stars["x_tile"] = tile_stars["x"] - x0
    tile_stars["y_tile"] = tile_stars["y"] - y0

    stem = f"r136_{tid:03d}_x{x0}_y{y0}"

    img_path = ROOT / "images" / split / f"{stem}.png"
    lbl_path = ROOT / "labels" / split / f"{stem}.txt"
    meta_path = ROOT / "metadata" / split / f"{stem}.csv"

    Image.fromarray(crop).save(img_path)

    label_lines = []

    for _, star in tile_stars.iterrows():
        cx, cy, bw, bh = clip_box(
            float(star["x_tile"]),
            float(star["y_tile"]),
            BOX_SIZE_PX,
            TILE_SIZE
        )

        if bw > 0 and bh > 0:
            label_lines.append(yolo_line(cx, cy, bw, bh, TILE_SIZE))

    lbl_path.write_text("\n".join(label_lines))

    tile_stars[
        ["HTTP", "RAdeg", "DEdeg", "F555mag", "x_tile", "y_tile"]
    ].to_csv(meta_path, index=False)

    records.append({
        "stem": stem,
        "split": split,
        "n_stars": len(tile_stars),
        "image": str(img_path),
        "labels": str(lbl_path),
        "metadata": str(meta_path),
    })

dataset_index = pd.DataFrame(records)

print(dataset_index.groupby("split")["n_stars"].agg(["count", "sum", "mean", "min", "max"]))
dataset_index.head()


In [ ]:

# ============================================================
# 17. Visualizar um tile e seu ground truth
# ============================================================

example = dataset_index.sort_values("n_stars", ascending=False).iloc[0]

img = np.asarray(Image.open(example["image"]))
meta = pd.read_csv(example["metadata"])

plt.figure(figsize=(9, 9))
plt.imshow(img, origin="upper")

plt.scatter(
    meta["x_tile"],
    TILE_SIZE - 1 - meta["y_tile"],  # ajuste apenas para a convenção visual do imshow
    s=22,
    facecolors="none",
    edgecolors="tab:red",
    linewidths=0.6,
)

plt.title(
    f"Tile mais denso — {example['split']} — "
    f"{len(meta)} estrelas catalogadas"
)
plt.xlabel("x [pixel]")
plt.ylabel("y [pixel]")
plt.show()



## 18. Arquivo `data.yaml` para Ultralytics YOLO

A classe será única:

```text
0 = star
```

Para esse tipo de campo, alguns parâmetros do YOLO precisam de atenção:

- `max_det` deve ser muito maior que o padrão, pois um tile pode conter centenas ou milhares de estrelas.
- A supressão de não-máximos (NMS) pode eliminar objetos muito próximos. Na inferência usaremos um `iou` relativamente alto.
- Desabilitamos `mosaic` e `mixup` neste baseline para não criar padrões artificiais de crowding.
- Evitamos alterações HSV, pois as imagens são essencialmente intensidade astronômica em uma única banda.


In [ ]:

# ============================================================
# 18. Criar data.yaml
# ============================================================

yaml_path = ROOT / "data.yaml"

config = {
    "path": str(ROOT),
    "train": "images/train",
    "val": "images/val",
    "names": {0: "star"},
}

with open(yaml_path, "w") as f:
    yaml.safe_dump(config, f, sort_keys=False)

print(yaml_path.read_text())



## 19. Treinar YOLO11

Começamos com **YOLO11s**. O objetivo aqui não é buscar imediatamente o maior modelo, mas medir se a abordagem consegue aprender o padrão de fontes pontuais e onde o desempenho degrada com crowding.

Se o Colab estiver sem GPU, ative em:

`Ambiente de execução → Alterar tipo de ambiente de execução → T4 GPU`


In [ ]:

# ============================================================
# 19. Treinamento
# ============================================================

from ultralytics import YOLO

model = YOLO("yolo11s.pt")

train_results = model.train(
    data=str(yaml_path),
    epochs=30,
    imgsz=TILE_SIZE,
    batch=4,
    device=0,          # troque para "cpu" se necessário
    workers=2,
    seed=SEED,

    # Campo astronômico: não usar augmentations que alterem cor.
    hsv_h=0.0,
    hsv_s=0.0,
    hsv_v=0.0,

    # Flips são fisicamente aceitáveis para o problema de detecção.
    fliplr=0.5,
    flipud=0.5,

    # Evitar composições artificiais de campos.
    mosaic=0.0,
    mixup=0.0,

    # Muitas fontes por imagem.
    max_det=5000,

    project="/content/runs_tarantula",
    name="yolo11s_r136",
)


In [ ]:

# ============================================================
# 20. Validação YOLO
# ============================================================

best_weights = Path("/content/runs_tarantula/yolo11s_r136/weights/best.pt")
best_model = YOLO(str(best_weights))

metrics = best_model.val(
    data=str(yaml_path),
    imgsz=TILE_SIZE,
    conf=0.05,
    iou=0.80,       # NMS mais permissivo para objetos muito próximos
    max_det=5000,
    device=0,
)

print(metrics)



## 21. Avaliação científica baseada no centro da estrela

mAP é útil, mas não é suficiente para astronomia.

Como nosso objeto físico é uma fonte pontual, avaliaremos também se o **centro da detecção** ficou suficientemente próximo da posição catalogada.

Usaremos um pareamento guloso por confiança dentro de `MATCH_RADIUS_PX`. Cada estrela do catálogo só pode ser associada a uma detecção.

Isso nos permite calcular:

- **TP:** detecção associada a uma estrela catalogada.
- **FP:** detecção sem estrela correspondente.
- **FN:** estrela catalogada que não foi detectada.
- **Precision**
- **Recall / Completeness**
- **Erro posicional**
- **Completeness × magnitude**


In [ ]:

# ============================================================
# 21. Função de pareamento centro-a-centro
# ============================================================

def greedy_match(pred_xy, pred_conf, gt_xy, radius_px):
    pred_xy = np.asarray(pred_xy, dtype=float)
    pred_conf = np.asarray(pred_conf, dtype=float)
    gt_xy = np.asarray(gt_xy, dtype=float)

    if len(gt_xy) == 0:
        return {
            "tp": 0,
            "fp": len(pred_xy),
            "fn": 0,
            "matched_gt": np.zeros(0, dtype=bool),
            "matched_distances": [],
        }

    matched_gt = np.zeros(len(gt_xy), dtype=bool)
    matched_distances = []

    if len(pred_xy) == 0:
        return {
            "tp": 0,
            "fp": 0,
            "fn": len(gt_xy),
            "matched_gt": matched_gt,
            "matched_distances": [],
        }

    order = np.argsort(-pred_conf)
    tree = cKDTree(gt_xy)

    tp = 0
    fp = 0

    for idx in order:
        dist, j = tree.query(pred_xy[idx], k=1)

        if dist <= radius_px and not matched_gt[j]:
            matched_gt[j] = True
            tp += 1
            matched_distances.append(float(dist))
        else:
            fp += 1

    fn = int((~matched_gt).sum())

    return {
        "tp": tp,
        "fp": fp,
        "fn": fn,
        "matched_gt": matched_gt,
        "matched_distances": matched_distances,
    }


In [ ]:

# ============================================================
# 22. Inferência e métricas em todos os tiles de validação
# ============================================================

all_eval = []
all_magnitudes = []
all_matched = []
all_distances = []

val_rows = dataset_index[dataset_index["split"] == "val"].reset_index(drop=True)

for _, row in val_rows.iterrows():
    result = best_model.predict(
        source=row["image"],
        imgsz=TILE_SIZE,
        conf=0.05,
        iou=0.80,
        max_det=5000,
        device=0,
        verbose=False,
    )[0]

    meta = pd.read_csv(row["metadata"])
    gt_xy = meta[["x_tile", "y_tile"]].to_numpy(dtype=float)

    if result.boxes is None or len(result.boxes) == 0:
        pred_xy = np.empty((0, 2), dtype=float)
        pred_conf = np.empty(0, dtype=float)
    else:
        xyxy = result.boxes.xyxy.detach().cpu().numpy()
        pred_conf = result.boxes.conf.detach().cpu().numpy()

        pred_xy = np.column_stack([
            (xyxy[:, 0] + xyxy[:, 2]) / 2.0,
            (xyxy[:, 1] + xyxy[:, 3]) / 2.0,
        ])

    m = greedy_match(
        pred_xy=pred_xy,
        pred_conf=pred_conf,
        gt_xy=gt_xy,
        radius_px=MATCH_RADIUS_PX,
    )

    all_eval.append({
        "tile": row["stem"],
        "tp": m["tp"],
        "fp": m["fp"],
        "fn": m["fn"],
        "n_gt": len(gt_xy),
        "n_pred": len(pred_xy),
    })

    all_distances.extend(m["matched_distances"])

    if len(meta):
        all_magnitudes.extend(meta["F555mag"].to_numpy(dtype=float))
        all_matched.extend(m["matched_gt"].astype(bool))

eval_df = pd.DataFrame(all_eval)

TP = int(eval_df["tp"].sum())
FP = int(eval_df["fp"].sum())
FN = int(eval_df["fn"].sum())

precision = TP / (TP + FP) if TP + FP else np.nan
recall = TP / (TP + FN) if TP + FN else np.nan
f1 = 2 * precision * recall / (precision + recall) if precision + recall else np.nan

print("TP:", TP)
print("FP:", FP)
print("FN:", FN)
print(f"Precision: {precision:.4f}")
print(f"Recall / Completeness: {recall:.4f}")
print(f"F1: {f1:.4f}")

if all_distances:
    print(f"Erro posicional mediano: {np.median(all_distances):.3f} px")
    print(f"Erro posicional P90: {np.percentile(all_distances, 90):.3f} px")


In [ ]:

# ============================================================
# 23. Completeness em função da magnitude F555W
# ============================================================

mag = np.asarray(all_magnitudes, dtype=float)
matched = np.asarray(all_matched, dtype=bool)

bins = np.arange(
    math.floor(np.nanmin(mag)),
    math.ceil(np.nanmax(mag)) + 0.5,
    0.5
)

rows = []

for lo, hi in zip(bins[:-1], bins[1:]):
    in_bin = (mag >= lo) & (mag < hi)
    n = int(in_bin.sum())

    if n == 0:
        continue

    comp = float(matched[in_bin].mean())

    rows.append({
        "mag_center": (lo + hi) / 2.0,
        "n_stars": n,
        "completeness": comp,
    })

completeness_df = pd.DataFrame(rows)
completeness_df


In [ ]:

# ============================================================
# 24. Curva de completeness x magnitude
# ============================================================

plt.figure(figsize=(9, 5))
plt.plot(
    completeness_df["mag_center"],
    completeness_df["completeness"],
    marker="o",
)
plt.ylim(0, 1.05)
plt.xlabel("Magnitude F555W")
plt.ylabel("Completeness / Recall")
plt.title("YOLO — completude em função da magnitude estelar")
plt.grid(alpha=0.25)
plt.show()


In [ ]:

# ============================================================
# 25. Desempenho em função da densidade do tile
# ============================================================

tile_metrics = eval_df.merge(
    dataset_index[["stem", "n_stars"]],
    left_on="tile",
    right_on="stem",
    how="left",
)

tile_metrics["recall"] = (
    tile_metrics["tp"] /
    (tile_metrics["tp"] + tile_metrics["fn"]).replace(0, np.nan)
)

plt.figure(figsize=(8, 5))
plt.scatter(tile_metrics["n_stars"], tile_metrics["recall"], s=60)
plt.xlabel("Estrelas catalogadas no tile")
plt.ylabel("Recall / Completeness")
plt.ylim(0, 1.05)
plt.title("Impacto do crowding no desempenho do YOLO")
plt.grid(alpha=0.25)
plt.show()

tile_metrics.sort_values("n_stars", ascending=False)


In [ ]:

# ============================================================
# 26. Visualizar predições em um tile de validação
# ============================================================

example_val = val_rows.sort_values("n_stars", ascending=False).iloc[0]

pred = best_model.predict(
    source=example_val["image"],
    imgsz=TILE_SIZE,
    conf=0.05,
    iou=0.80,
    max_det=5000,
    device=0,
    verbose=False,
)[0]

annotated_bgr = pred.plot()
annotated_rgb = annotated_bgr[..., ::-1]

plt.figure(figsize=(10, 10))
plt.imshow(annotated_rgb)
plt.title(
    f"YOLO — predições no tile de validação mais denso "
    f"({example_val['n_stars']} fontes de catálogo)"
)
plt.axis("off")
plt.show()



# Interpretação e próximos experimentos

Este notebook é deliberadamente um **baseline**. Alguns pontos devem ser tratados com cuidado:

### 1. YOLO usa caixas, mas estrelas são fontes pontuais

Uma estrela em uma imagem HST é melhor descrita pela **PSF** do sistema óptico do que por uma caixa delimitadora. YOLO é útil como primeira comparação, mas arquiteturas center-based ou mapas de probabilidade podem ser mais naturais para o problema.

### 2. Crowding e NMS

Em R136, estrelas vizinhas podem ter PSFs e caixas fortemente sobrepostas. A etapa de **Non-Maximum Suppression (NMS)** do YOLO pode suprimir uma estrela real por considerá-la uma detecção duplicada.

Por isso usamos:

- `iou=0.80`
- `max_det=5000`

Esses valores devem ser estudados como hiperparâmetros, e não assumidos como definitivos.

### 3. Limite de magnitude

Ao usar `MAG_LIMIT = 25.5`, estrelas mais fracas ficam fora do ground truth do experimento, embora possam permanecer visíveis na imagem. Isso cria **label noise**.

Um experimento importante será repetir o pipeline para diferentes limites:

```text
F555W < 23
F555W < 24
F555W < 25
F555W < 26
F555W < 27
```

e medir como precision/recall mudam.

### 4. Astrometria

Estamos cruzando um produto Hubble HAP com o catálogo HTTP por RA/Dec. A sobreposição visual deve ser verificada antes do treino. Para um estudo mais rigoroso, devemos medir e corrigir qualquer offset global entre as duas soluções astrométricas.

### 5. Comparação científica

O próximo benchmark recomendado é:

```text
HTTP catalog
      |
      +------ YOLO11
      |
      +------ DAOStarFinder
      |
      +------ PSF photometry / fitting
               |
               v
      Precision / Completeness
      erro posicional
      magnitude limite
      desempenho x crowding
```

### 6. Dataset maior

Para evitar que o estudo fique restrito a um único cutout, devemos montar três regimes dentro de 30 Doradus:

```text
baixa densidade
      ↓
densidade intermediária
      ↓
R136 — densidade extrema
```

Esse desenho permitirá testar explicitamente:

**crowding ↑ → completeness ?**

---

## Referências principais

- Sabbi, E. et al. (2016), *Hubble Tarantula Treasury Project III: Photometric Catalog and Resulting Constraints on the Progression of Star Formation in the 30 Doradus Region*, ApJS 222, 11. DOI: `10.3847/0067-0049/222/1/11`.
- MAST HTTP: `https://archive.stsci.edu/hlsp/http`
- VizieR HTTP: `https://cdsarc.cds.unistra.fr/viz-bin/cat/J/ApJS/222/11`
- MAST HAPCut / Astroquery: `https://astroquery.readthedocs.io/en/stable/mast/mast_cut.html`
- NASA/STScI R136: `https://science.nasa.gov/asset/hubble/star-cluster-r136-in-nebula-30-doradus/`
